In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Res
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [4]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_Res()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_16").output,
    outputs=model.output
)

In [8]:
l_label=[3, 11, 18, 27, 34, 43, 50, 59, 66]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_2/Res-18/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_2/Res-18/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_2/Res-18/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_2/Res-18/layer_cache/base
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_2: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_4: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_6: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_8: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_10: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_12: outputs (20000, 2048), labels (20000,)
[Saved] re_lu_14: outputs (20000, 1024), labels (20000,)
[Saved] re_lu_16: outputs (20000, 1024), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data_2/Res-18/layer_cache/gauss/0
[Saved] re_lu: outputs (20000, 32768), labels (20000,)
[Saved] re_lu_2: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_4: outputs (20000, 8192), labels (20000,)
[Saved] re_lu_6: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_8: outputs (20000, 4096), labels (20000,)
[Saved] re_lu_10: outputs (20000, 2048), labels (20000,)
[Saved] re_l

In [11]:
CACHE_DIR = "./RES-18/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_2/Res-18/layer_cache/base")


==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 17541
xi>=0 count: 15514
xi>=0 count: 13898
xi>=0 count: 13192
xi>=0 count: 13119
xi>=0 count: 11346
xi>=0 count: 11883
xi>=0 count: 13699
xi>=0 count: 16420

==== split 0, threshold=2000 ====

==== split 1, threshold=4000 ====

==== split 2, threshold=6000 ====

==== split 3, threshold=8000 ====

==== split 4, threshold=10000 ====

==== split 5, threshold=12000 ====

==== split 6, threshold=14000 ====

==== split 7, threshold=16000 ====

==== split 8, threshold=18000 ====
xi>=0 count: 17956
xi>=0 count: 16291
xi>=0 count: 14870
xi>=0 count: 14292
xi>=0 count: 14803
xi>=0 count: 12841
xi>=0 count: 12152
xi>=0 count: 13605
xi>=0 count: 16255

==== split 0, thr

In [13]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_2/Res-18/layer_cache/base")

In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_2/Res-18/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.89134514 0.70396809 0.6181496  0.60673508 0.61738999 0.4959037
 0.60689694 0.69895883 0.86531419]
Layer 1
accuracy: [0.90773341 0.73122302 0.65262619 0.62187495 0.6681397  0.54936211
 0.58715936 0.68234039 0.80684409]
Layer 2
accuracy: [0.90392846 0.71415351 0.63348937 0.61585259 0.67580834 0.54422548
 0.54639896 0.6737718  0.85414991]
Layer 3
accuracy: [0.94951105 0.80093255 0.69678702 0.66780667 0.72097818 0.67066939
 0.67774489 0.75134688 0.79716439]
Layer 4
accuracy: [0.96690494 0.8533387  0.73008771 0.70241109 0.75920165 0.71453233
 0.70758325 0.76095715 0.83177425]
Layer 5
accuracy: [0.96649924 0.94643258 0.8776258  0.83691758 0.82024711 0.83044945
 0.82269108 0.84126222 0.9166844 ]
Layer 6
accuracy: [0.96819476 0.96797583 0.90601725 0.87317972 0.8632528  0.87057222
 0.84876763 0.87780218 0.92346678]
Layer 7
accuracy: [0.97167782 0.97462705 0.94703692 0.91637966 0.90318649 0.87775965
 0.89722235 0.92879904 0.91442091]
Layer 8
accuracy: [0.97252101 0.97436822 

In [16]:
compute_stats(base)

(array([[0.73782094, 0.57334292, 0.72372332],
        [0.76386087, 0.61312558, 0.69211461],
        [0.75052378, 0.61196214, 0.69144023],
        [0.81574354, 0.68648475, 0.74208539],
        [0.85011045, 0.72538169, 0.76677155],
        [0.93018588, 0.82920471, 0.86021257],
        [0.94739595, 0.86900158, 0.88334553],
        [0.96444726, 0.8991086 , 0.91348077],
        [0.94517468, 0.83233507, 0.83334268],
        [1.        , 1.        , 1.        ]]),
 array([[0.11407151, 0.0549303 , 0.10694182],
        [0.10667348, 0.04888382, 0.08995183],
        [0.11336203, 0.05378887, 0.12625844],
        [0.10370433, 0.02441853, 0.04919069],
        [0.09670717, 0.02442094, 0.05086719],
        [0.03805775, 0.00686237, 0.04064499],
        [0.02925929, 0.00420207, 0.03074668],
        [0.01236971, 0.01602807, 0.01290826],
        [0.03998683, 0.02004612, 0.14342393],
        [0.        , 0.        , 0.        ]]))

In [17]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/Res-18/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.56188451 0.42504267 0.39518515 0.45720002 0.56647956 0.39464433
 0.71382273 0.82463718 0.92007914]
Layer 1
accuracy: [0.95496359 0.77677188 0.68079593 0.63419087 0.64880831 0.54905264
 0.61360295 0.72484166 0.87144809]
Layer 2
accuracy: [0.92970386 0.7318574  0.64794846 0.61593731 0.65899059 0.53648413
 0.5592049  0.69433714 0.86197495]
Layer 3
accuracy: [0.96185446 0.80855044 0.69273262 0.64243282 0.68724312 0.64842068
 0.66603714 0.75701612 0.83014063]
Layer 4
accuracy: [0.97941262 0.83138529 0.71750456 0.66397255 0.69790292 0.66630562
 0.6743959  0.75443845 0.84784392]
Layer 5
accuracy: [0.98379672 0.88169001 0.7848912  0.73439404 0.72312516 0.7167844
 0.74487241 0.80757293 0.91784935]
Layer 6
accuracy: [0.98292898 0.8999228  0.80293972 0.75419489 0.72763764 0.71981996
 0.75298193 0.81908908 0.91854414]
Layer 7
accuracy: [0.98376365 0.8999043  0.83584738 0.77170713 0.73713481 0.70281297
 0.75419995 0.8365424  0.89477874]
Layer 8
accuracy: [0.98375804 0.89700166 

In [18]:
compute_stats(gauss)

(array([[0.46013178, 0.47414706, 0.81951302],
        [0.80467134, 0.61211448, 0.73693639],
        [0.77076207, 0.60338041, 0.70474839],
        [0.82069873, 0.65704381, 0.74937664],
        [0.84437466, 0.67466842, 0.75758239],
        [0.88375087, 0.72385867, 0.81934146],
        [0.89460083, 0.73439363, 0.82815328],
        [0.9054945 , 0.73785303, 0.8286048 ],
        [0.89344496, 0.69474992, 0.79718275],
        [0.92802194, 0.77699699, 0.87817189]]),
 array([[0.0713751 , 0.07144758, 0.08428175],
        [0.11336308, 0.04247645, 0.10635228],
        [0.11921714, 0.05274922, 0.12658837],
        [0.11090216, 0.01951148, 0.0679637 ],
        [0.10781416, 0.01526789, 0.07359681],
        [0.07867076, 0.01146189, 0.07641182],
        [0.07223275, 0.01418168, 0.07213488],
        [0.06069066, 0.02898378, 0.0575314 ],
        [0.07343857, 0.04027015, 0.03148702],
        [0.04908516, 0.0204439 , 0.05693167]]))

In [19]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_2/Res-18/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.69805563 0.53609862 0.4764988  0.50790803 0.59816588 0.41108198
 0.71150607 0.82091747 0.91966315]
Layer 1
accuracy: [0.82033809 0.65708266 0.58875312 0.57933743 0.64871085 0.5063582
 0.47844341 0.56395689 0.6549394 ]
Layer 2
accuracy: [0.88836839 0.68954425 0.61404221 0.59408613 0.66623091 0.52753236
 0.51770369 0.64261922 0.82792427]
Layer 3
accuracy: [0.96205816 0.8087858  0.69441666 0.66082085 0.70205546 0.66760726
 0.68811405 0.77126958 0.83692458]
Layer 4
accuracy: [0.97596921 0.84015989 0.72321396 0.68595306 0.72841737 0.68821183
 0.69833066 0.76720762 0.84449031]
Layer 5
accuracy: [0.9790426  0.91169867 0.8407685  0.78188448 0.77317771 0.77141616
 0.77883294 0.82286495 0.91977439]
Layer 6
accuracy: [0.9777315  0.94210888 0.86607071 0.80382978 0.7970348  0.79105288
 0.79299037 0.85729202 0.92068757]
Layer 7
accuracy: [0.98076437 0.93751037 0.90935058 0.83789692 0.82358965 0.7849662
 0.81642794 0.88170334 0.90541536]
Layer 8
accuracy: [0.9811917  0.93646047 0

In [20]:
compute_stats(salt)

(array([[0.56980834, 0.50563843, 0.8174686 ],
        [0.68797672, 0.57666029, 0.56706912],
        [0.72833412, 0.59495709, 0.66127902],
        [0.82242533, 0.6760947 , 0.76586178],
        [0.84733849, 0.70192278, 0.76902473],
        [0.9091739 , 0.77846872, 0.84224017],
        [0.92632289, 0.80125642, 0.85782656],
        [0.94026646, 0.81312913, 0.87079107],
        [0.92303317, 0.75895895, 0.81837263],
        [0.9669006 , 0.8708484 , 0.93664209]]),
 array([[0.09255732, 0.07608688, 0.08537188],
        [0.09692935, 0.05667379, 0.07133454],
        [0.11587706, 0.05790995, 0.12831263],
        [0.10766694, 0.02062438, 0.0584817 ],
        [0.10272032, 0.01897567, 0.06037792],
        [0.05695914, 0.00684582, 0.05753433],
        [0.04663544, 0.00644882, 0.04971939],
        [0.03104017, 0.02404992, 0.03501691],
        [0.05256878, 0.0270003 , 0.07535264],
        [0.02065814, 0.01354564, 0.02936564]]))

In [21]:
SAVE_FILE='e-Res-18.pkl'

In [22]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [2]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [3]:
SAVE_FILE='e-Res-18.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [4]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [5]:
mean_var

{'base': {'mean': array([0.67829573, 0.68970036, 0.68464205, 0.74810456, 0.78075456,
         0.87320105, 0.89991435, 0.92567888, 0.87028414, 1.        ]),
  'std': array([0.12122735, 0.10524495, 0.11740865, 0.08598225, 0.08288154,
         0.0532264 , 0.0420466 , 0.03127455, 0.10162778, 0.        ]),
  'min': array([0.4959037 , 0.54936211, 0.54422548, 0.66780667, 0.70241109,
         0.82024711, 0.84876763, 0.87775965, 0.63093374, 1.        ]),
  'max': array([0.89134514, 0.90773341, 0.90392846, 0.94951105, 0.96690494,
         0.96649924, 0.96819476, 0.97462705, 0.97436822, 1.        ])},
 'gauss': {'mean': array([0.58459729, 0.7179074 , 0.69296362, 0.74237306, 0.75887516,
         0.80898367, 0.81904925, 0.82398411, 0.79512588, 0.86106361]),
  'std': array([0.18273741, 0.12254043, 0.12548102, 0.10126548, 0.10275554,
         0.09147479, 0.08865594, 0.08547346, 0.09618133, 0.07726972]),
  'min': array([0.39450879, 0.552944  , 0.53347373, 0.64067045, 0.65977251,
         0.71343322, 0